# End-to-End NYISO DART Virtual Bidding Pipeline Orchestrator

Executes all four research notebooks sequentially in dependency order (`01` $\rightarrow$ `04`), ensuring each stage writes its cached `.parquet` benchmark artifacts to `nyiso_data/merged_parquet/` before the next notebook loads them:

1. **`01_baseline_evaluation.ipynb`** $\rightarrow$ Builds/loads `merged_parquet` cache & exports `stage1_baseline_results_NYC.parquet`
2. **`02_dmdc_static_forensics.ipynb`** $\rightarrow$ Exports `dmdc_static_oos_results.parquet`
3. **`03_dmdc_iterative_scheduled_refit.ipynb`** $\rightarrow$ Exports `dmdc_scheduled_oos_results.parquet`
4. **`04_sspoc_sparse_sensor_placement.ipynb`** $\rightarrow$ Runs `SSPOC` ablation & cross-paradigm `30-Day Refit SSPOC vs. DMDc` benchmark


In [2]:
import time
from pathlib import Path
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

# Resolve notebooks directory regardless of whether Jupyter was launched from root or notebooks/
NOTEBOOKS_DIR = Path("notebooks").resolve() if Path("notebooks").exists() else Path(".").resolve()

PIPELINE_NOTEBOOKS = [
    "01_baseline_evaluation.ipynb",
    "02_dmdc_static_forensics.ipynb",
    "03_dmdc_iterative_scheduled_refit.ipynb",
    "04_sspoc_sparse_sensor_placement.ipynb",
]

# Verify all notebooks exist before starting
missing = [nb for nb in PIPELINE_NOTEBOOKS if not (NOTEBOOKS_DIR / nb).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing notebooks in {NOTEBOOKS_DIR}: {missing}\n"
        "Update PIPELINE_NOTEBOOKS to match your exact .ipynb filenames."
    )

# Optional: Save executed copies with refreshed outputs in-place (set False to leave source notebooks untouched)
SAVE_EXECUTED_IN_PLACE = True
KERNEL_NAME = "python3"

summary = []
pipeline_start = time.perf_counter()

for step, nb_name in enumerate(PIPELINE_NOTEBOOKS, start=1):
    nb_path = NOTEBOOKS_DIR / nb_name
    print(f"[{step}/{len(PIPELINE_NOTEBOOKS)}] Running {nb_name} ...")
    step_start = time.perf_counter()

    with nb_path.open("r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)
        nbformat.validator.normalize(nb)

    client = NotebookClient(
        nb,
        timeout=1800,  # 30-minute per-cell safety limit
        kernel_name=KERNEL_NAME,
        resources={"metadata": {"path": str(NOTEBOOKS_DIR)}},
    )

    try:
        client.execute()
    except CellExecutionError as exc:
        elapsed = time.perf_counter() - step_start
        print(f"FAILED: {nb_name} after {elapsed:.1f}s")
        raise RuntimeError(f"Pipeline halted at {nb_name}") from exc

    if SAVE_EXECUTED_IN_PLACE:
        with nb_path.open("w", encoding="utf-8") as f:
            nbformat.write(nb, f)

    elapsed = time.perf_counter() - step_start
    summary.append({"Step": step, "Notebook": nb_name, "Status": "OK", "Runtime (s)": round(elapsed, 2)})
    print(f"    Completed {nb_name} in {elapsed:.1f}s\n")

total_elapsed = time.perf_counter() - pipeline_start
print(f"=== Full Pipeline Complete in {total_elapsed:.1f}s ===")
for row in summary:
    print(f"  {row['Step']}. {row['Notebook']:42s} | {row['Status']} | {row['Runtime (s)']:6.2f}s")


[1/4] Running 01_baseline_evaluation.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


    Completed 01_baseline_evaluation.ipynb in 27.7s

[2/4] Running 02_dmdc_static_forensics.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


    Completed 02_dmdc_static_forensics.ipynb in 147.2s

[3/4] Running 03_dmdc_iterative_scheduled_refit.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


    Completed 03_dmdc_iterative_scheduled_refit.ipynb in 15.1s

[4/4] Running 04_sspoc_sparse_sensor_placement.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[CACHE HIT] Loading sparse sensor ablation results (13beb82a3c0f...)...
[CACHE HIT] Loading sparse sensor ablation results (28e4e849f0a0...)...
    Completed 04_sspoc_sparse_sensor_placement.ipynb in 39.2s

=== Full Pipeline Complete in 229.2s ===
  1. 01_baseline_evaluation.ipynb               | OK |  27.68s
  2. 02_dmdc_static_forensics.ipynb             | OK | 147.24s
  3. 03_dmdc_iterative_scheduled_refit.ipynb    | OK |  15.12s
  4. 04_sspoc_sparse_sensor_placement.ipynb     | OK |  39.15s
